# VieNeu-TTS v3 Turbo — Fine-tune LoRA tăng UTMOS

**Mục tiêu**: fine-tune VieNeu-TTS trên một giọng nam để tăng UTMOS (hiện tại 2.024), giữ WER/CER tốt và RTF nhanh.

**Baseline trước fine-tune**: WER 0.0199 | CER 0.0101 | UTMOS 2.024 | SIM-o 0.7742 | RTF 0.1640

**Dataset**: VIVOS (`AILAB-VNUHCM/vivos`), 15 giờ, 46 speaker, không gated.

**GPU**: Kaggle T4 (16GB) — LoRA cần ~6GB VRAM, bắt buộc `--no-bf16` vì T4 không hỗ trợ bf16.

**Cảnh báo về UTMOS**: VIVOS là dataset ASR thu ở 16kHz, trong khi VieNeu-TTS xuất 48kHz. UTMOS nhạy với dải tần cao, mà nâng 16kHz lên 48kHz không tạo thêm thông tin tần số cao. Fine-tune trên VIVOS có thể làm UTMOS giảm thay vì tăng. Transcript VIVOS cũng viết hoa và không có dấu câu nên ngữ điệu học được sẽ kém hơn. Muốn tăng UTMOS thật thì cần dataset làm riêng cho TTS, ví dụ VieNeu-TTS-140h (gated, phải xin quyền).

**Lưu ý**: nghe ở Cell 3b để chọn giọng nam, điền vào `SPEAKER_ID` ở Cell 4, còn lại chạy thẳng.

In [ ]:
# ============================================================
# Cell 1: CÀI ĐẶT
# ============================================================
# KHÔNG cài torch thủ công trên Kaggle. Cài đè torch sẽ làm hỏng
# bản dựng CUDA có sẵn và gây DeferredCudaCallError.
!pip install -q "transformers>=4.50,<4.58" peft accelerate datasets soundfile librosa
!pip install -q vieneu jiwer pandas matplotlib sherpa-onnx

import torch
print("torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CẢNH BÁO: không thấy GPU.")
    print("Nếu torch bị hỏng do pip cài đè, chạy cell sau rồi restart kernel:")
    print("  !pip uninstall -y torch torchaudio torchvision")
    print("  !pip install torch torchaudio torchvision")

In [ ]:
# ============================================================
# Cell 2: CLONE REPO (lấy script fine-tune)
# ============================================================
# Dùng đường dẫn tuyệt đối để chạy lại cell này nhiều lần không tạo ra
# repo lồng trong repo (VieNeu-TTS/VieNeu-TTS/VieNeu-TTS/...).
import os, subprocess

REPO_DIR = "/kaggle/working/VieNeu-TTS"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/pnnbao97/VieNeu-TTS.git", REPO_DIR],
        check=True)
else:
    print("Repo đã có, bỏ qua clone.")

%cd /kaggle/working/VieNeu-TTS
print("cwd:", os.getcwd())

In [ ]:
# ============================================================
# Cell 2b: ĐĂNG NHẬP HUGGINGFACE (tuỳ chọn)
# ============================================================
# VIVOS không gated nên bỏ qua cell này vẫn chạy được.
# Chỉ cần nếu sau này đổi sang VieNeu-TTS-140h (gated):
#   1. Vào https://huggingface.co/datasets/pnnbao-ump/VieNeu-TTS-140h
#      đăng nhập và bấm chấp nhận điều khoản.
#   2. Tạo token Read tại https://huggingface.co/settings/tokens
#   3. Kaggle: Add-ons > Secrets > thêm secret nhãn HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login, whoami

    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Đã đăng nhập:", whoami()["name"])
except Exception as e:
    print(f"Chưa đăng nhập ({type(e).__name__}) — không sao, VIVOS không cần.")

In [ ]:
# ============================================================
# Cell 3: TẢI VIVOS + LIỆT KÊ SPEAKER
# ============================================================
from datasets import load_dataset, Audio
from huggingface_hub import HfApi, hf_hub_download
from collections import Counter

VIVOS = "AILAB-VNUHCM/vivos"
PARQUET_REV = "refs/convert/parquet"

# datasets >= 3.0 bỏ hỗ trợ loading script, mà VIVOS vẫn dùng vivos.py nên
# load_dataset("AILAB-VNUHCM/vivos") sẽ lỗi. Dùng bản parquet mà HuggingFace
# tự convert sẵn ở nhánh refs/convert/parquet.
files = HfApi().list_repo_files(VIVOS, repo_type="dataset", revision=PARQUET_REV)
train_files = sorted(f for f in files if f.endswith(".parquet") and "/train/" in f)
assert train_files, f"Không thấy parquet train. Các file có: {files[:20]}"
print(f"Tải {len(train_files)} file parquet...")

local = [hf_hub_download(VIVOS, f, repo_type="dataset", revision=PARQUET_REV)
         for f in train_files]
dataset = load_dataset("parquet", data_files=local, split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print(f"Tổng: {len(dataset)} mẫu")
print("Các cột:", list(dataset.features), "\n")

COL_SPEAKER, COL_TEXT = "speaker_id", "sentence"
for c in (COL_SPEAKER, COL_TEXT):
    assert c in dataset.features, f"Không có cột '{c}'. Xem danh sách cột ở trên."

# VIVOS không có nhãn giới tính nên phải nghe để chọn giọng nam (Cell 3b)
speaker_counts = Counter(dataset[COL_SPEAKER])
print(f"Số speaker: {len(speaker_counts)}\n")
print("Top 12 speaker theo số clip:")
for spk, count in speaker_counts.most_common(12):
    print(f"  {spk}: {count} clips")

In [ ]:
# ============================================================
# Cell 3b: NGHE THỬ ĐỂ CHỌN GIỌNG NAM
# ============================================================
from IPython.display import Audio as PlayAudio, display

TOP = [spk for spk, _ in speaker_counts.most_common(8)]

first_idx = {}
for i, spk in enumerate(dataset[COL_SPEAKER]):
    if spk in TOP and spk not in first_idx:
        first_idx[spk] = i
    if len(first_idx) == len(TOP):
        break

for spk in TOP:
    s = dataset[first_idx[spk]]
    print(f"{spk} — {speaker_counts[spk]} clips — {s[COL_TEXT][:60]}")
    display(PlayAudio(s["audio"]["array"], rate=s["audio"]["sampling_rate"]))

In [ ]:
# ============================================================
# Cell 4: CHỌN SPEAKER + EXPORT DATA
# ============================================================
# ===== SỬA SPEAKER_ID SAU KHI NGHE Ở CELL 3b =====
SPEAKER_ID = "???"  # vd "VIVOSSPK12" — chọn giọng NAM có nhiều clip nhất
# =================================================

import os
import soundfile as sf

# Đọc riêng cột speaker rồi select theo chỉ số. Dùng dataset.filter() sẽ giải mã
# audio của cả 11660 dòng chỉ để so sánh một chuỗi, mất khoảng 2 phút.
all_speakers = dataset[COL_SPEAKER]
idx = [i for i, s in enumerate(all_speakers) if s == SPEAKER_ID]

if not idx:
    avail = sorted(set(all_speakers))
    raise ValueError(
        f"Không có clip nào của '{SPEAKER_ID}'.\n"
        f"Chạy Cell 3b để nghe và chọn giọng nam.\n"
        f"Các ID hợp lệ: {avail[:15]}{' ...' if len(avail) > 15 else ''}"
    )

speaker_data = dataset.select(idx)
print(f"Speaker {SPEAKER_ID}: {len(speaker_data)} clips")

output_dir = "finetune/dataset"
audio_dir = os.path.join(output_dir, "raw_audio")
os.makedirs(audio_dir, exist_ok=True)

total_sec = 0.0
sr = None
with open(os.path.join(output_dir, "metadata.csv"), "w", encoding="utf-8") as f:
    f.write("file_name|text\n")
    for i, sample in enumerate(speaker_data):
        fname = f"clip_{i:05d}.wav"
        arr = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]
        sf.write(os.path.join(audio_dir, fname), arr, sr)
        total_sec += len(arr) / sr
        # VIVOS viết hoa toàn bộ và không có dấu câu, hạ về chữ thường
        text = sample[COL_TEXT].strip().lower()
        f.write(f"{fname}|{text}\n")

print(f"Exported {len(speaker_data)} clips, {total_sec/60:.1f} phút vào {output_dir}")
print(f"Sample rate: {sr} Hz")

# Giải phóng RAM
del dataset, speaker_data
import gc; gc.collect()

In [ ]:
# ============================================================
# Cell 5: PREPARE DATASET (encode audio bằng MOSS codec)
# ============================================================
!python finetune/prepare_dataset.py --dataset-dir finetune/dataset

import os, glob
print("\nFile sinh ra trong finetune/dataset:")
for f in sorted(glob.glob("finetune/dataset/*")):
    if os.path.isfile(f):
        print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")

assert os.path.exists("finetune/dataset/train.parquet"), \
    "Không có train.parquet. Đọc output phía trên xem prepare_dataset.py lỗi gì."
print("\ntrain.parquet OK.")

In [ ]:
# ============================================================
# Cell 5b: CHẠY THỬ 10 BƯỚC TRƯỚC KHI TRAIN THẬT
# ============================================================
# Mất 2-3 phút. Nếu bước này hỏng thì train đầy đủ cũng hỏng y hệt,
# phát hiện sớm đỡ mất một tiếng chờ rồi mới biết.
!python finetune/train_lora.py \
    --data finetune/dataset/train.parquet \
    --run smoke_test \
    --no-bf16 \
    --max-steps 10 \
    --save-every 5 \
    --r 16 --alpha 32 \
    --batch-size 2 \
    --grad-accum 2 \
    --grad-checkpoint

import os
ok = os.path.isdir("finetune/output/smoke_test/adapter")
print()
print("Pipeline chạy được, sang Cell 6." if ok else
      "THẤT BẠI: không sinh ra adapter/. Đọc lỗi phía trên trước khi chạy Cell 6.")

In [ ]:
# ============================================================
# Cell 6: TRAIN LoRA (T4: bắt buộc --no-bf16)
# ============================================================
!python finetune/train_lora.py \
    --data finetune/dataset/train.parquet \
    --run male_voice \
    --no-bf16 \
    --epochs 3 \
    --lr 2e-4 \
    --r 16 \
    --alpha 32 \
    --batch-size 4 \
    --grad-accum 4 \
    --grad-checkpoint \
    --save-every 200 \
    --merge

import os
print()
for sub in ["adapter", "merged"]:
    p = f"finetune/output/male_voice/{sub}"
    print(f"  [{'x' if os.path.isdir(p) else ' '}] {p}")

assert os.path.exists("finetune/output/male_voice/merged/config.json"), \
    "Chưa có merged/config.json. Train hoặc merge thất bại, đọc log phía trên."
print("\nMerged model OK.")

In [ ]:
# ============================================================
# Cell 7: ĐÓNG GÓI VOICE
# ============================================================
# make_voice.py cần một clip sạch dài 3-8 giây để lấy speaker embedding.
# VIVOS có nhiều câu rất ngắn nên chọn clip dài phù hợp thay vì lấy bừa clip_00000.
import glob, soundfile as sf

REF_CLIP, ref_dur = None, 0.0
for w in sorted(glob.glob("finetune/dataset/raw_audio/*.wav")):
    info = sf.info(w)
    d = info.frames / info.samplerate
    if 3.0 <= d <= 8.0 and d > ref_dur:
        REF_CLIP, ref_dur = w, d

assert REF_CLIP, "Không có clip nào dài 3-8 giây. Đổi speaker hoặc nới điều kiện."
print(f"Dùng làm ref: {REF_CLIP}  ({ref_dur:.1f}s)\n")

!python finetune/make_voice.py \
    --audio {REF_CLIP} \
    --name "Male Voice" \
    --gender male \
    --out finetune/output/male_voice/merged

In [ ]:
# ============================================================
# Cell 8: LOAD MODEL ĐÃ FINE-TUNE + WARM-UP
# ============================================================
import time, os, json, glob, re, unicodedata
import numpy as np
import soundfile as sf
import librosa
import pandas as pd
from pathlib import Path
from vieneu import Vieneu
from IPython.display import Audio

tts = Vieneu(mode="v3turbo", backbone_repo="finetune/output/male_voice/merged", device="cuda")

# Warm-up
_ = tts.infer("Xin chào.", voice="Male Voice")
print("Model fine-tuned loaded")

# Test nghe thử
audio = tts.infer("Xin chào, đây là giọng nói đã được tinh chỉnh thành công.", voice="Male Voice")
tts.save(audio, "/kaggle/working/test_finetuned.wav")
Audio("/kaggle/working/test_finetuned.wav")

In [ ]:
# ============================================================
# Cell 9: TẬP TEST (49 câu, giống notebook đánh giá cũ)
# ============================================================
D = []
def add(pid, cat, text, spoken=None, alts=None, stress=False):
    D.append({"id": pid, "category": cat, "text": text,
              "spoken": spoken if spoken is not None else text,
              "alts": alts or [], "stress": stress})

add("everyday_01","everyday","Hôm nay thời tiết khá đẹp, rất thích hợp để đi dạo công viên.")
add("everyday_02","everyday","Cà phê sữa đá là thức uống yêu thích của rất nhiều người Việt Nam.")
add("everyday_03","everyday","Chiều nay mình phải ghé siêu thị mua ít đồ dùng cho gia đình.")
add("everyday_04","everyday","Con đường trước nhà vừa được sửa lại nên đi êm hơn hẳn.")
add("everyday_05","everyday","Bữa tối nay mẹ nấu canh chua cá lóc và thịt kho tiêu.")
add("everyday_06","everyday","Mình thường dậy lúc sáu giờ sáng để tập thể dục trước khi đi làm.")
add("everyday_07","everyday","Cuối tuần này cả nhà dự định về quê thăm ông bà nội.")
add("everyday_08","everyday","Quán phở đầu ngõ mở cửa từ sớm và lúc nào cũng đông khách.")
add("everyday_09","everyday","Trời trở lạnh rồi, nhớ mặc thêm áo khoác khi ra ngoài nhé.")
add("everyday_10","everyday","Tôi vừa đọc xong một cuốn sách rất hay về thói quen hằng ngày.")
add("question_01","question","Bạn đã ăn cơm chưa? Nếu chưa thì mình cùng đi ăn nhé.")
add("question_02","question","Tại sao hôm nay trời lại mưa to như vậy nhỉ?")
add("question_03","question","Cho mình hỏi đường ra bến xe miền Đông đi lối nào ạ?")
add("question_04","question","Anh có biết cuộc họp sáng mai bắt đầu lúc mấy giờ không?")
add("question_05","question","Em đã nộp báo cáo cho phòng nhân sự chưa vậy?")
add("emotion_01","emotion","Wow, đội tuyển vừa ghi bàn thắng đẹp quá!")
add("emotion_02","emotion","Trời ơi, xem pháo hoa tối qua đẹp không thể tin nổi!")
add("emotion_03","emotion","Chúc mừng bạn nhé, cuối cùng thì công sức cũng được đền đáp rồi!")
add("emotion_04","emotion","Đừng lo, mọi chuyện rồi sẽ ổn thôi, tôi luôn ở đây với bạn.")
add("emotion_05","emotion","Hạnh phúc đôi khi chỉ là được về nhà ăn cơm cùng gia đình.")
add("number_01","number","Hóa đơn hôm nay là 1.250.000 đồng, giảm 15% so với tuần trước.",
    "Hóa đơn hôm nay là một triệu hai trăm năm mươi nghìn đồng, giảm mười lăm phần trăm so với tuần trước.")
add("number_02","number","Nhiệt độ ngoài trời hiện tại là 32 độ C, độ ẩm 78%.",
    "Nhiệt độ ngoài trời hiện tại là ba mươi hai độ xê, độ ẩm bảy mươi tám phần trăm.")
add("number_03","number","Số điện thoại liên hệ của tôi là 0912 345 678.",
    "Số điện thoại liên hệ của tôi là không chín một hai ba bốn năm sáu bảy tám.")
add("number_04","number","Doanh thu quý ba đạt 47,5 tỷ đồng, tăng 8,2% so với cùng kỳ.",
    "Doanh thu quý ba đạt bốn mươi bảy phẩy năm tỷ đồng, tăng tám phẩy hai phần trăm so với cùng kỳ.")
add("number_05","number","Lớp tôi có 42 sinh viên, trong đó 18 bạn nữ và 24 bạn nam.",
    "Lớp tôi có bốn mươi hai sinh viên, trong đó mười tám bạn nữ và hai mươi bốn bạn nam.")
add("abbrev_01","abbrev","TP.HCM và Hà Nội là hai thành phố lớn nhất Việt Nam.",
    "Thành phố Hồ Chí Minh và Hà Nội là hai thành phố lớn nhất Việt Nam.")
add("abbrev_02","abbrev","UBND tỉnh vừa ban hành quyết định mới về quy hoạch đô thị.",
    "Ủy ban nhân dân tỉnh vừa ban hành quyết định mới về quy hoạch đô thị.")
add("abbrev_03","abbrev","Công ty TNHH Ánh Dương tuyển thêm 20 nhân viên kinh doanh.",
    "Công ty trách nhiệm hữu hạn Ánh Dương tuyển thêm hai mươi nhân viên kinh doanh.")
add("long_01","long","Dự án tốt nghiệp của tôi tập trung vào việc đánh giá và triển khai mô hình chuyển văn bản thành giọng nói cho tiếng Việt, nhằm phục vụ mục đích nghiên cứu và ứng dụng thực tế.")
add("long_02","long","Hệ thống sẽ tự động gửi thông báo tới địa chỉ đã đăng ký ngay khi đơn hàng được xác nhận, và người dùng có thể theo dõi toàn bộ hành trình vận chuyển ngay trên ứng dụng di động.")
add("long_03","long","Trước khi đưa mô hình vào môi trường thực tế, nhóm nghiên cứu đã tiến hành kiểm thử trên nhiều tập dữ liệu khác nhau, đo lường cẩn thận từng chỉ số và ghi nhận toàn bộ trường hợp lỗi.")
add("news_01","news","Bộ Giáo dục và Đào tạo vừa công bố phương án thi tốt nghiệp cho năm học tới.")
add("news_02","news","Giá xăng trong nước tiếp tục được điều chỉnh tăng từ chiều nay.")
add("news_03","news","Ngành du lịch đặt mục tiêu đón lượng khách quốc tế cao kỷ lục trong năm nay.")
add("narration_01","narration","Ngày xửa ngày xưa, ở một ngôi làng nhỏ ven sông, có một cậu bé tên là Tí.")
add("narration_02","narration","Buổi chiều hôm ấy, gió thổi qua cánh đồng lúa chín vàng rực cả một vùng.")
add("narration_03","narration","Tiếng chuông chùa vang lên từ phía xa, ngân dài trong buổi sớm tinh mơ.")
add("tech_01","tech","Nhóm mình dùng Python và Docker để triển khai hệ thống.",
    alts=["Nhóm mình dùng pai thon và đốc cơ để triển khai hệ thống."], stress=True)
add("tech_02","tech","Máy chủ bị quá tải do lượng truy cập tăng đột biến trong giờ cao điểm.")
add("code_mix_01","code_mix","Tôi mới học xong khóa Python cơ bản trên Coursera.",
    alts=["Tôi mới học xong khóa pai thon cơ bản trên cơ rờ sợ ra."], stress=True)
add("code_mix_02","code_mix","Repo này dùng Docker và deploy lên GitHub Actions.",
    alts=["Ri pâu này dùng đốc cơ và đì plói lên gịt hắp ách sần."], stress=True)
add("code_mix_03","code_mix","Nhớ check email trước khi họp lúc 9 giờ sáng nhé.",
    alts=["Nhớ chéc i meo trước khi họp lúc chín giờ sáng nhé."], stress=True)
add("code_mix_04","code_mix","Wifi ở quán này chậm quá, chắc phải restart lại router.",
    alts=["Quai phai ở quán này chậm quá, chắc phải ri sờ tạc lại rao tơ."], stress=True)
add("code_mix_05","code_mix","Cho mình xin cái link Google Drive để tải file lên nhé.",
    alts=["Cho mình xin cái linh gu gồ đờ rai vơ để tải phai lên nhé."], stress=True)
add("code_mix_06","code_mix","Video này được share trên Facebook mấy triệu lượt xem.",
    alts=["Vi đi ô này được se trên phây búc mấy triệu lượt xem."], stress=True)
add("code_mix_07","code_mix","App bị crash liên tục sau khi update bản mới.",
    alts=["Áp bị cờ rát liên tục sau khi ắp đết bản mới."], stress=True)
add("code_mix_08","code_mix","Laptop của tôi hết pin nên phải sạc bằng sạc dự phòng.",
    alts=["Láp tóp của tôi hết pin nên phải sạc bằng sạc dự phòng."], stress=True)
add("misc_01","misc","Trân trọng cảm ơn quý khách đã tin tưởng và sử dụng dịch vụ của chúng tôi.")
add("misc_02","misc","Chúc quý vị và gia đình một năm mới an khang thịnh vượng.")

PROMPTS = {d["id"]: d for d in D}
from collections import Counter
print(len(D), "câu |", dict(Counter(d["category"] for d in D)))

In [ ]:
# ============================================================
# Cell 10: SINH AUDIO + ĐO RTF
# ============================================================
AUDIO_DIR = "/kaggle/working/audio_finetuned"
os.makedirs(AUDIO_DIR, exist_ok=True)

timings = {}
done = fail = 0

for pid, p in PROMPTS.items():
    out = f"{AUDIO_DIR}/{pid}.wav"
    try:
        t0 = time.time()
        audio = tts.infer(p["text"], voice="Male Voice")
        dt = time.time() - t0
        audio = np.asarray(audio, dtype=np.float32).squeeze()
        sf.write(out, audio, 48000)
        timings[pid] = {"gen_s": round(dt, 3),
                        "audio_s": round(len(audio) / 48000, 3),
                        "rtf": round(dt / max(len(audio) / 48000, 1e-6), 4)}
        done += 1
    except Exception as e:
        fail += 1
        print(f"lỗi {pid}: {e}")
    if (done + fail) % 10 == 0:
        print(f"  ...{done+fail}/{len(PROMPTS)}")

rtfs = [v["rtf"] for v in timings.values()]
print(f"\nXong. Sinh {done}, lỗi {fail}")
print(f"RTF trung bình: {np.mean(rtfs):.4f}")

In [ ]:
# ============================================================
# Cell 11: TẢI ASR + BUILD RECOGNIZER
# ============================================================
import sherpa_onnx, urllib.request, tarfile

ASR_DIR = "/kaggle/working/asr_models"
os.makedirs(ASR_DIR, exist_ok=True)

BASE = "https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models"
PKGS = {
    "zipformer_vi_70k": "sherpa-onnx-zipformer-vi-2025-04-20",
    "zipformer_vi_30m": "sherpa-onnx-zipformer-vi-30M-int8-2026-02-09",
}

for key, name in PKGS.items():
    dst = f"{ASR_DIR}/{name}"
    if os.path.isdir(dst):
        print("đã có:", name); continue
    print("tải:", name, "...")
    arc = f"{ASR_DIR}/{name}.tar.bz2"
    urllib.request.urlretrieve(f"{BASE}/{name}.tar.bz2", arc)
    with tarfile.open(arc, 'r:bz2') as tf:
        tf.extractall(ASR_DIR)
    os.remove(arc)
    print("  xong")

def _find(d, prefix):
    files = [f for f in os.listdir(d) if f.startswith(prefix) and f.endswith(".onnx")]
    if not files:
        raise FileNotFoundError(f"Không thấy {prefix}*.onnx trong {d}")
    int8 = sorted([f for f in files if "int8" in f])
    return os.path.join(d, int8[0] if int8 else sorted(files)[0])

def build_recognizer(pkg_key):
    name = PKGS[pkg_key]
    d = f"{ASR_DIR}/{name}"
    return sherpa_onnx.OfflineRecognizer.from_transducer(
        encoder=_find(d, "encoder"), decoder=_find(d, "decoder"),
        joiner=_find(d, "joiner"), tokens=os.path.join(d, "tokens.txt"),
        num_threads=2, sample_rate=16000, feature_dim=80, decoding_method="greedy_search")

def transcribe(recognizer, wav_path):
    wave, _ = librosa.load(wav_path, sr=16000, mono=True)
    s = recognizer.create_stream()
    s.accept_waveform(16000, wave.astype(np.float32))
    recognizer.decode_stream(s)
    return s.result.text

ASR_MAIN  = build_recognizer("zipformer_vi_70k")
ASR_CROSS = build_recognizer("zipformer_vi_30m")
print("ASR loaded")

In [ ]:
# ============================================================
# Cell 12: NORMALIZER + SCORING
# ============================================================
_ONES = ["không","một","hai","ba","bốn","năm","sáu","bảy","tám","chín"]
def _read_3(n, full):
    tr, ch, dv = n // 100, (n // 10) % 10, n % 10
    out = []
    if tr or full: out += [_ONES[tr], "trăm"]
    if ch == 0 and dv:
        if tr or full: out.append("linh")
        out.append(_ONES[dv])
    elif ch == 1:
        out.append("mười")
        if dv == 5: out.append("lăm")
        elif dv: out.append(_ONES[dv])
    elif ch > 1:
        out += [_ONES[ch], "mươi"]
        if dv == 1: out.append("mốt")
        elif dv == 4: out.append("tư")
        elif dv == 5: out.append("lăm")
        elif dv: out.append(_ONES[dv])
    return " ".join(out).strip()
_SCALES = ["", "nghìn", "triệu", "tỷ"]
def num_to_words(n):
    if n == 0: return "không"
    if n < 0:  return "âm " + num_to_words(-n)
    groups = []
    while n: groups.append(n % 1000); n //= 1000
    parts = []
    for i in range(len(groups) - 1, -1, -1):
        if groups[i] == 0: continue
        parts.append(_read_3(groups[i], full=(i != len(groups) - 1)))
        if _SCALES[i % 4]: parts.append(_SCALES[i % 4])
    return " ".join(parts)
def digits_to_words(s):
    return " ".join(_ONES[int(c)] for c in s if c.isdigit())

ABBREV = {"tp.hcm":"thành phố hồ chí minh","tphcm":"thành phố hồ chí minh",
    "ubnd":"ủy ban nhân dân","thpt":"trung học phổ thông",
    "tnhh":"trách nhiệm hữu hạn","gs":"giáo sư","ts":"tiến sĩ",
    "pgs":"phó giáo sư","ths":"thạc sĩ","cn":"cử nhân",
    "btc":"ban tổ chức","hđqt":"hội đồng quản trị","gdp":"tổng sản phẩm quốc nội"}
_PUNCT = re.compile(r"[^\w\s]", flags=re.UNICODE)
_WS = re.compile(r"\s+")
_NUM = re.compile(r"\d[\d.,]*")

def normalize_v1(s, expand_numbers=True):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = s.replace("<unk>", " ").replace("\u2026", " ")
    for k in sorted(ABBREV, key=len, reverse=True):
        s = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", " " + ABBREV[k] + " ", s)
    s = s.replace("%", " phần trăm ")
    if expand_numbers:
        def _rep(m):
            raw = m.group(0)
            if re.fullmatch(r"\d{1,3}(\.\d{3})+", raw):
                return " " + num_to_words(int(raw.replace(".", ""))) + " "
            if re.fullmatch(r"\d{1,3}(,\d{3})+", raw):
                return " " + num_to_words(int(raw.replace(",", ""))) + " "
            if re.fullmatch(r"\d+,\d+", raw):
                a, b = raw.split(",")
                return f" {num_to_words(int(a))} phẩy {digits_to_words(b)} "
            if re.fullmatch(r"\d+\.\d+", raw):
                a, b = raw.split(".")
                return f" {num_to_words(int(a))} chấm {digits_to_words(b)} "
            digits = re.sub(r"\D", "", raw)
            if not digits: return " "
            if len(digits) > 6: return " " + digits_to_words(digits) + " "
            return " " + num_to_words(int(digits)) + " "
        s = _NUM.sub(_rep, s)
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

import jiwer
def score_pair(ref_variants, hyp):
    h = normalize_v1(hyp)
    best = None
    for ref in ref_variants:
        r = normalize_v1(ref)
        if not r: continue
        w, c = jiwer.wer(r, h), jiwer.cer(r, h)
        if best is None or w < best[0]:
            best = (w, c, r, h)
    return best if best else (1.0, 1.0, "", h)
print("Normalizer + Scoring OK")

In [ ]:
# ============================================================
# Cell 13: WER
# ============================================================
rows = []
wavs = sorted(glob.glob(f"{AUDIO_DIR}/*.wav"))
print(f"Chấm WER: {len(wavs)} file...")

for w in wavs:
    pid = Path(w).stem
    if pid not in PROMPTS: continue
    p = PROMPTS[pid]
    rv = [p["spoken"]] + list(p.get("alts", []))
    h1 = transcribe(ASR_MAIN, w)
    h2 = transcribe(ASR_CROSS, w)
    w1, c1, ref_used, hyp1 = score_pair(rv, h1)
    w2, c2, _, _ = score_pair(rv, h2)
    rows.append({"prompt_id": pid, "category": p["category"],
                 "stress": p["stress"], "ref": ref_used,
                 "hyp_main": hyp1, "wer": w1, "cer": c1,
                 "wer_cross": w2, "wer_agree": min(w1, w2)})

df_wer = pd.DataFrame(rows)
core = df_wer[~df_wer.stress]
print(f"\nWER:  {core.wer.mean():.4f}")
print(f"CER:  {core.cer.mean():.4f}")
print(f"\nWER theo nhóm:")
print(core.groupby("category")["wer"].mean().round(4).sort_values().to_string())

In [ ]:
# ============================================================
# Cell 14: UTMOS
# ============================================================
predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
predictor = predictor.to("cuda").eval()

utmos_rows = []
for w in sorted(glob.glob(f"{AUDIO_DIR}/*.wav")):
    wave, _ = librosa.load(w, sr=16000, mono=True)
    with torch.no_grad():
        s = predictor(torch.from_numpy(wave).unsqueeze(0).to("cuda"), 16000).item()
    utmos_rows.append({"prompt_id": Path(w).stem, "utmos": s})

df_utmos = pd.DataFrame(utmos_rows)
print(f"UTMOS trung bình: {df_utmos.utmos.mean():.3f} ± {df_utmos.utmos.std():.3f}")
print(f"UTMOS trước fine-tune: 2.024")

In [ ]:
# ============================================================
# Cell 15: SIM-o
# ============================================================
from transformers import AutoFeatureExtractor, WavLMForXVector
import torch.nn.functional as F

_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
_sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to("cuda").eval()

def embed(wav_path):
    wave, _ = librosa.load(wav_path, sr=16000, mono=True)
    inp = _fe([wave], sampling_rate=16000, return_tensors="pt", padding=True)
    inp = {k: v.to("cuda") for k, v in inp.items()}
    with torch.no_grad():
        e = _sv(**inp).embeddings
    return F.normalize(e, dim=-1)

# Cùng clip đã dùng để enroll voice ở Cell 7
ref_emb = embed(REF_CLIP)
sim_rows = []
for w in sorted(glob.glob(f"{AUDIO_DIR}/*.wav")):
    sim = float(torch.cosine_similarity(ref_emb, embed(w)).item())
    sim_rows.append({"prompt_id": Path(w).stem, "sim_o": sim})

df_sim = pd.DataFrame(sim_rows)
print(f"SIM-o trung bình: {df_sim.sim_o.mean():.4f} ± {df_sim.sim_o.std():.4f}")

In [ ]:
# ============================================================
# Cell 16: BÁO CÁO TỔNG HỢP
# ============================================================
core = df_wer[~df_wer.stress]

print("="*60)
print("BÁO CÁO: VieNeu-TTS v3 Turbo (Fine-tuned)")
print("="*60)
print(f"Model        : VieNeu-TTS v3 Turbo + LoRA fine-tune")
print(f"Speaker      : {SPEAKER_ID} (male, từ VIVOS 16kHz)")
print(f"Dataset      : {len(PROMPTS)} câu test")
print(f"GPU          : {torch.cuda.get_device_name(0)}")

print(f"\n{'Metric':<20} {'Trước FT':>12} {'Sau FT':>12}")
print("-"*46)
print(f"{'WER':<20} {'0.0199':>12} {core.wer.mean():>12.4f}")
print(f"{'CER':<20} {'0.0101':>12} {core.cer.mean():>12.4f}")
print(f"{'UTMOS':<20} {'2.024':>12} {df_utmos.utmos.mean():>12.3f}")
print(f"{'SIM-o':<20} {'0.7742':>12} {df_sim.sim_o.mean():>12.4f}")
print(f"{'RTF (single)':<20} {'0.1640':>12} {np.mean(rtfs):>12.4f}")